<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Generative-Deep-Learning/blob/main/scripts/Art_gallery_exhibition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# @title **PCA Art Gallery Exhibition** (빈 공간 클릭 시 이미지 재현 지원)
# ============================================================
# MNIST 2D 시각화 + 클릭하면 이미지 보기 (Google Colab용)
#
# - PCA는 선형 변환이라 inverse_transform()으로 "역변환"이 가능함
#   -> 데이터가 없는 빈 공간을 클릭해도, 그 좌표가 원래 어떤 이미지였을지
#      실제로 계산해서 재현할 수 있음
# - t-SNE는 비선형이라 역변환이 존재하지 않음
#   -> 빈 공간을 클릭하면 "가장 가까운 실제 데이터의 이미지"를 근사로 보여줌
#      (진짜 생성이 아니라 최근접 이웃이라는 점에 유의)
#
# 아래 내용을 Colab 셀에 그대로 붙여넣고 실행하세요.
# ============================================================

# ---- 설정값 (필요시 수정) ----
N_SAMPLES = 2000       # 사용할 샘플 개수 (너무 크면 t-SNE가 느려짐)
METHOD = "pca"         # "pca" (역변환으로 진짜 재현 가능) 또는 "tsne" (최근접 이웃으로만 근사)

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import Output, VBox
from IPython.display import display
from scipy.spatial import cKDTree

# ---- 1. MNIST 데이터 로드 (Colab에는 tensorflow가 기본 설치되어 있음) ----
from tensorflow.keras.datasets import mnist

(x_train, y_train), (_, _) = mnist.load_data()
x_train = x_train.astype(np.float32) / 255.0

idx = np.random.choice(len(x_train), min(N_SAMPLES, len(x_train)), replace=False)
images = x_train[idx]          # (N, 28, 28)
labels = y_train[idx]          # (N,)
flat = images.reshape(len(images), -1)

# ---- 2. 2차원으로 축소 ----
pca_model = None  # PCA를 쓸 때만 inverse_transform용으로 저장

if METHOD == "pca":
    from sklearn.decomposition import PCA
    pca_model = PCA(n_components=2, random_state=42)
    coords = pca_model.fit_transform(flat)
else:  # tsne
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    pre = PCA(n_components=50, random_state=42).fit_transform(flat)
    coords = TSNE(n_components=2, random_state=42, init="pca", perplexity=30).fit_transform(pre)

# t-SNE의 경우 빈 공간 클릭 시 최근접 이웃을 찾기 위한 KD-Tree
tree = cKDTree(coords)

# ---- 3. "빈 공간"도 클릭되게 하기 위한 투명 격자(heatmap) 준비 ----
pad_x = (coords[:, 0].max() - coords[:, 0].min()) * 0.1
pad_y = (coords[:, 1].max() - coords[:, 1].min()) * 0.1
x_min, x_max = coords[:, 0].min() - pad_x, coords[:, 0].max() + pad_x
y_min, y_max = coords[:, 1].min() - pad_y, coords[:, 1].max() + pad_y

GRID_RES = 80
xs_grid = np.linspace(x_min, x_max, GRID_RES)
ys_grid = np.linspace(y_min, y_max, GRID_RES)

background_grid = go.Heatmap(
    x=xs_grid, y=ys_grid, z=np.zeros((GRID_RES, GRID_RES)),
    colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],  # 완전 투명
    showscale=False,
    hoverinfo="none",  # "skip"으로 두면 클릭 이벤트까지 막히므로 반드시 "none" 사용
    opacity=0,
)

data_scatter = go.Scattergl(
    x=coords[:, 0],
    y=coords[:, 1],
    mode="markers",
    marker=dict(
        size=6,
        color=labels,
        colorscale="Rainbow",
        showscale=True,
        colorbar=dict(title="숫자"),
    ),
    text=[f"숫자: {l}" for l in labels],
    hoverinfo="text",
)

# 순서 중요: 배경(heatmap)을 먼저, 마커(scatter)를 나중에 그려서
# 마커 위를 클릭하면 마커가, 빈 공간을 클릭하면 heatmap이 클릭을 받는다.
fig = go.FigureWidget(data=[background_grid, data_scatter])
fig.update_layout(
    title=f"MNIST {METHOD.upper()} 2D Embedding",
    width=650,
    height=600,
)

# ---- 4. 클릭 시 이미지를 보여줄 출력 영역 ----
out = Output()

def show_image(image, title):
    with out:
        out.clear_output(wait=True)
        plt.figure(figsize=(2.5, 2.5))
        plt.imshow(image, cmap="gray", vmin=0, vmax=1)
        plt.title(title)
        plt.axis("off")
        plt.show()

def on_click_marker(trace, points, state):
    # 실제 데이터 점을 클릭 -> 그 점의 원본 이미지를 그대로 표시
    if not points.point_inds:
        return
    i = points.point_inds[0]
    show_image(images[i], f"숫자: {labels[i]}")

def on_click_empty(trace, points, state):
    # 빈 공간을 클릭한 경우
    if not points.point_inds:
        return
    x_click = points.xs[0]
    y_click = points.ys[0]

    if METHOD == "pca":
        # PCA는 선형이라 inverse_transform으로 "진짜" 재현이 가능
        point_2d = np.array([[x_click, y_click]])
        recon = pca_model.inverse_transform(point_2d).reshape(28, 28)
        recon = np.clip(recon, 0, 1)
        show_image(recon, f"PCA Inversed Image\n({x_click:.1f}, {y_click:.1f})")
    else:
        # t-SNE는 역변환이 없으므로 가장 가까운 실제 이미지로 근사
        dist, i = tree.query([x_click, y_click])
        show_image(images[i], f"K-Nearest Neighbor (Number {labels[i]})\n거리: {dist:.2f}")

fig.data[0].on_click(on_click_empty)   # heatmap (배경)
fig.data[1].on_click(on_click_marker)  # scatter (마커)

# ---- 5. 화면에 표시 ----
display(VBox([fig, out]))

    'data': [{'colorscale': [[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
      …

In [25]:
# @title **Autoencoder  Art Gellery Exhibition**
# ============================================================
# Autoencoder 기반 MNIST 2D 잠재공간 시각화 (숫자 5, 8만 사용)
# - 인코더로 5/8 이미지를 2차원 latent space에 투영
# - 산점도의 점을 클릭하면 원본과 재생성 이미지를 나란히, 빈 공간을 클릭하면
#   그 좌표에서 디코더가 새로 생성한 이미지를 보여줌
#
# * matplotlib의 클릭 이벤트(ipympl)는 Colab의 최신 matplotlib 버전과
#   충돌해 백엔드 등록 오류가 나는 경우가 많아, 여기서는 별도 백엔드가
#   필요 없는 Plotly FigureWidget의 클릭 콜백을 사용합니다.
#
# Google Colab 셀에 그대로 붙여넣고 실행하세요. (GPU 런타임 권장)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import tensorflow as tf
from tensorflow.keras import layers, models
from ipywidgets import Output, VBox
from IPython.display import display

# ---- 설정값 ----
LATENT_DIM = 2
EPOCHS = 20
BATCH_SIZE = 256
N_PLOT_SAMPLES = 3000   # 산점도에 표시할 샘플 수 (많으면 느려짐)

# ---- 1. 데이터 로드 (5번과 8번만 사용) ----
TARGET_DIGITS = [5, 8]

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

train_mask = np.isin(y_train, TARGET_DIGITS)
test_mask = np.isin(y_test, TARGET_DIGITS)
x_train, y_train = x_train[train_mask], y_train[train_mask]
x_test, y_test = x_test[test_mask], y_test[test_mask]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train_flat = x_train.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

# ---- 2. 오토인코더 정의 (병목층 = 2차원) ----
# 인코더: 784 -> ... -> 2
encoder_input = layers.Input(shape=(784,))
h = layers.Dense(256, activation="relu")(encoder_input)
h = layers.Dense(64, activation="relu")(h)
latent = layers.Dense(LATENT_DIM, activation="linear", name="latent")(h)
encoder = models.Model(encoder_input, latent, name="encoder")

# 디코더: 2 -> ... -> 784
decoder_input = layers.Input(shape=(LATENT_DIM,))
h = layers.Dense(64, activation="relu")(decoder_input)
h = layers.Dense(256, activation="relu")(h)
decoder_output = layers.Dense(784, activation="sigmoid")(h)
decoder = models.Model(decoder_input, decoder_output, name="decoder")

# 오토인코더 = 인코더 + 디코더
autoencoder = models.Model(encoder_input, decoder(encoder(encoder_input)), name="autoencoder")
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

# ---- 3. 학습 ----
autoencoder.fit(
    x_train_flat, x_train_flat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_test_flat, x_test_flat),
    verbose=1,
)

# ---- 4. 시각화용 샘플을 잠재공간으로 인코딩 ----
n_plot = min(N_PLOT_SAMPLES, len(x_test_flat))
idx = np.random.choice(len(x_test_flat), n_plot, replace=False)
z = encoder.predict(x_test_flat[idx], verbose=0)
labels_plot = y_test[idx]
images_for_plot = x_test[idx]  # 클릭 시 원본과 비교해서 보여주기 위해 보관

# ---- 5. "빈 공간"도 클릭되게 하기 위한 투명 격자(heatmap) 준비 ----
# Scattergl의 on_click은 마커 위를 클릭했을 때만 반응하므로,
# 화면 전체를 덮는 투명한 heatmap을 하나 더 깔아서 그 격자의 클릭을 받는다.
pad_x = (z[:, 0].max() - z[:, 0].min()) * 0.1
pad_y = (z[:, 1].max() - z[:, 1].min()) * 0.1
x_min, x_max = z[:, 0].min() - pad_x, z[:, 0].max() + pad_x
y_min, y_max = z[:, 1].min() - pad_y, z[:, 1].max() + pad_y

GRID_RES = 80  # 격자를 촘촘하게 할수록 클릭 정밀도가 올라가지만 너무 크면 느려짐
xs_grid = np.linspace(x_min, x_max, GRID_RES)
ys_grid = np.linspace(y_min, y_max, GRID_RES)

background_grid = go.Heatmap(
    x=xs_grid, y=ys_grid, z=np.zeros((GRID_RES, GRID_RES)),
    colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],  # 완전 투명
    showscale=False,
    # 주의: hoverinfo="skip"으로 두면 호버뿐 아니라 클릭(on_click) 이벤트까지
    # 함께 막혀버려서 빈 공간 클릭이 전혀 반응하지 않게 된다.
    # "none"으로 두면 호버 텍스트만 안 보이고 클릭 이벤트는 정상적으로 들어온다.
    hoverinfo="none",
    opacity=0,
)

data_scatter = go.Scattergl(
    x=z[:, 0], y=z[:, 1],
    mode="markers",
    marker=dict(
        size=7,
        color=labels_plot,
        colorscale=[[0, "royalblue"], [1, "orangered"]],  # 5=파랑, 8=주황
        showscale=True,
        colorbar=dict(title="숫자", tickvals=TARGET_DIGITS),
    ),
    text=[f"Number: {l}" for l in labels_plot],
    hoverinfo="text",
)

# 순서 중요: 배경(heatmap)을 먼저, 마커(scatter)를 나중에 그려서
# 마커 위를 클릭하면 마커가, 빈 공간을 클릭하면 heatmap이 클릭을 받는다.
fig = go.FigureWidget(data=[background_grid, data_scatter])
fig.update_layout(
    title="Autoencoder 2D 잠재공간 (5 vs 8) — 점이든 빈 공간이든 클릭해보세요",
    width=650, height=600,
    xaxis_title="latent dim 1",
    yaxis_title="latent dim 2",
)

out = Output()

def show_reconstruction(x, y, original_image=None, label=None):
    z_point = np.array([[x, y]], dtype="float32")
    recon = decoder.predict(z_point, verbose=0).reshape(28, 28)

    with out:
        out.clear_output(wait=True)
        if original_image is not None:
            fig2, axes = plt.subplots(1, 2, figsize=(4.5, 2.5))
            axes[0].imshow(original_image, cmap="gray", vmin=0, vmax=1)
            axes[0].set_title("Original Number")
            axes[0].axis("off")
            axes[1].imshow(recon, cmap="gray", vmin=0, vmax=1)
            title = f"Generated (Number {label})" if label is not None else "Generation by AE"
            axes[1].set_title(title)
            axes[1].axis("off")
        else:
            fig2, ax = plt.subplots(figsize=(3, 3))
            ax.imshow(recon, cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"latent=({x:.2f}, {y:.2f})")
            ax.axis("off")
        plt.tight_layout()
        plt.show()

def on_click_marker(trace, points, state):
    # 실제 데이터 점을 클릭한 경우 -> 원본과 재생성 이미지를 함께 표시
    if not points.point_inds:
        return
    i = points.point_inds[0]
    show_reconstruction(z[i, 0], z[i, 1], original_image=images_for_plot[i], label=labels_plot[i])

def on_click_empty(trace, points, state):
    # 빈 공간을 클릭한 경우 -> 그 좌표만으로 디코더가 이미지 생성
    if not points.point_inds:
        return
    x_click = points.xs[0]
    y_click = points.ys[0]
    show_reconstruction(x_click, y_click)

fig.data[0].on_click(on_click_empty)   # heatmap (배경)
fig.data[1].on_click(on_click_marker)  # scatter (마커)

display(VBox([fig, out]))

Epoch 1/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.3669 - val_loss: 0.2556
Epoch 2/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 0.2465 - val_loss: 0.2311
Epoch 3/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - loss: 0.2232 - val_loss: 0.2193
Epoch 4/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.2161 - val_loss: 0.2161
Epoch 5/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.2133 - val_loss: 0.2142
Epoch 6/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.2117 - val_loss: 0.2128
Epoch 7/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.2102 - val_loss: 0.2115
Epoch 8/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.2085 - val_loss: 0.2109
Epoch 9/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.2074 - val_loss: 0.2090
Epoch 10/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.2060 - val_loss: 0.2082
Epoch 11/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.2051 - val_loss: 0.2071
Epoch 12/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.2

    'data': [{'colorscale': [[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
      …

In [26]:
# @title **Variational Autoencoder(VAE)  Art Gellery Exhibition**
# ============================================================
# Variational Autoencoder(VAE) 기반 MNIST 2D 잠재공간 시각화 (숫자 5, 8만 사용)
# - 인코더가 각 이미지를 하나의 점이 아니라 "분포(평균 z_mean, 분산 z_log_var)"로
#   투영하고, KL divergence로 그 분포가 표준정규분포에 가깝도록 강제함
#   -> 일반 Autoencoder보다 잠재공간이 연속적이고 빈틈이 적어,
#      데이터가 없는 빈 공간을 클릭해도 훨씬 자연스러운 이미지가 생성됨
# - 산점도의 점을 클릭하면 원본과 재생성 이미지를 나란히, 빈 공간을 클릭하면
#   그 좌표에서 디코더가 새로 생성한 이미지를 보여줌
#
# * matplotlib의 클릭 이벤트(ipympl)는 Colab의 최신 matplotlib 버전과
#   충돌해 백엔드 등록 오류가 나는 경우가 많아, 여기서는 별도 백엔드가
#   필요 없는 Plotly FigureWidget의 클릭 콜백을 사용합니다.
#
# Google Colab 셀에 그대로 붙여넣고 실행하세요. (GPU 런타임 권장)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import tensorflow as tf
from tensorflow.keras import layers, models
from ipywidgets import Output, VBox
from IPython.display import display

# ---- 설정값 ----
LATENT_DIM = 2
EPOCHS = 20
BATCH_SIZE = 256
N_PLOT_SAMPLES = 3000   # 산점도에 표시할 샘플 수 (많으면 느려짐)

# ---- 1. 데이터 로드 (5번과 8번만 사용) ----
TARGET_DIGITS = [5, 8]

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

train_mask = np.isin(y_train, TARGET_DIGITS)
test_mask = np.isin(y_test, TARGET_DIGITS)
x_train, y_train = x_train[train_mask], y_train[train_mask]
x_test, y_test = x_test[test_mask], y_test[test_mask]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train_flat = x_train.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

# ---- 2. VAE 정의 (병목층 = 2차원) ----

# 인코더: 784 -> ... -> (z_mean, z_log_var) -> 샘플링 -> z
encoder_input = layers.Input(shape=(784,))
h = layers.Dense(256, activation="relu")(encoder_input)
h = layers.Dense(64, activation="relu")(h)
z_mean = layers.Dense(LATENT_DIM, name="z_mean")(h)
z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(h)

def sampling(args):
    """재매개변수화 트릭(reparameterization trick):
    z = mu + sigma * epsilon (epsilon ~ N(0, I))"""
    z_mean_, z_log_var_ = args
    epsilon = tf.random.normal(shape=tf.shape(z_mean_))
    return z_mean_ + tf.exp(0.5 * z_log_var_) * epsilon

z_sample = layers.Lambda(sampling, output_shape=(LATENT_DIM,), name="z")([z_mean, z_log_var])
encoder = models.Model(encoder_input, [z_mean, z_log_var, z_sample], name="encoder")

# 디코더: 2 -> ... -> 784 (일반 오토인코더와 동일한 구조)
decoder_input = layers.Input(shape=(LATENT_DIM,))
h = layers.Dense(64, activation="relu")(decoder_input)
h = layers.Dense(256, activation="relu")(h)
decoder_output = layers.Dense(784, activation="sigmoid")(h)
decoder = models.Model(decoder_input, decoder_output, name="decoder")


class VAE(models.Model):
    """재구성 손실 + KL divergence 손실을 함께 최적화하는 VAE 래퍼 모델."""

    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.recon_loss_tracker = tf.keras.metrics.Mean(name="recon_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def call(self, inputs):
        z_mean_, z_log_var_, z_ = self.encoder(inputs)
        return self.decoder(z_)

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]
        with tf.GradientTape() as tape:
            z_mean_, z_log_var_, z_ = self.encoder(data)
            reconstruction = self.decoder(z_)

            recon_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.keras.backend.binary_crossentropy(data, reconstruction), axis=1
                )
            )
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var_ - tf.square(z_mean_) - tf.exp(z_log_var_), axis=1)
            )
            total_loss = recon_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "recon_loss": self.recon_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }


vae = VAE(encoder, decoder)
vae.compile(optimizer="adam")

# ---- 3. 학습 ----
vae.fit(
    x_train_flat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=1,
)

# ---- 4. 시각화용 샘플을 잠재공간으로 인코딩 ----
# 시각화/클릭 위치 표시에는 샘플링된 z가 아니라 안정적인 z_mean(평균)을 사용
n_plot = min(N_PLOT_SAMPLES, len(x_test_flat))
idx = np.random.choice(len(x_test_flat), n_plot, replace=False)
z_mean_pred, z_log_var_pred, _ = encoder.predict(x_test_flat[idx], verbose=0)
z = z_mean_pred
labels_plot = y_test[idx]
images_for_plot = x_test[idx]  # 클릭 시 원본과 비교해서 보여주기 위해 보관

# ---- 5. "빈 공간"도 클릭되게 하기 위한 투명 격자(heatmap) 준비 ----
# Scattergl의 on_click은 마커 위를 클릭했을 때만 반응하므로,
# 화면 전체를 덮는 투명한 heatmap을 하나 더 깔아서 그 격자의 클릭을 받는다.
pad_x = (z[:, 0].max() - z[:, 0].min()) * 0.1
pad_y = (z[:, 1].max() - z[:, 1].min()) * 0.1
x_min, x_max = z[:, 0].min() - pad_x, z[:, 0].max() + pad_x
y_min, y_max = z[:, 1].min() - pad_y, z[:, 1].max() + pad_y

GRID_RES = 80  # 격자를 촘촘하게 할수록 클릭 정밀도가 올라가지만 너무 크면 느려짐
xs_grid = np.linspace(x_min, x_max, GRID_RES)
ys_grid = np.linspace(y_min, y_max, GRID_RES)

background_grid = go.Heatmap(
    x=xs_grid, y=ys_grid, z=np.zeros((GRID_RES, GRID_RES)),
    colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],  # 완전 투명
    showscale=False,
    # 주의: hoverinfo="skip"으로 두면 호버뿐 아니라 클릭(on_click) 이벤트까지
    # 함께 막혀버려서 빈 공간 클릭이 전혀 반응하지 않게 된다.
    # "none"으로 두면 호버 텍스트만 안 보이고 클릭 이벤트는 정상적으로 들어온다.
    hoverinfo="none",
    opacity=0,
)

data_scatter = go.Scattergl(
    x=z[:, 0], y=z[:, 1],
    mode="markers",
    marker=dict(
        size=7,
        color=labels_plot,
        colorscale=[[0, "royalblue"], [1, "orangered"]],  # 5=파랑, 8=주황
        showscale=True,
        colorbar=dict(title="숫자", tickvals=TARGET_DIGITS),
    ),
    text=[f"Number: {l}" for l in labels_plot],
    hoverinfo="text",
)

# 순서 중요: 배경(heatmap)을 먼저, 마커(scatter)를 나중에 그려서
# 마커 위를 클릭하면 마커가, 빈 공간을 클릭하면 heatmap이 클릭을 받는다.
fig = go.FigureWidget(data=[background_grid, data_scatter])
fig.update_layout(
    title="VAE 2D 잠재공간 (5 vs 8) — 점이든 빈 공간이든 클릭해보세요",
    width=650, height=600,
    xaxis_title="latent dim 1",
    yaxis_title="latent dim 2",
)

out = Output()

def show_reconstruction(x, y, original_image=None, label=None):
    z_point = np.array([[x, y]], dtype="float32")
    recon = decoder.predict(z_point, verbose=0).reshape(28, 28)

    with out:
        out.clear_output(wait=True)
        if original_image is not None:
            fig2, axes = plt.subplots(1, 2, figsize=(4.5, 2.5))
            axes[0].imshow(original_image, cmap="gray", vmin=0, vmax=1)
            axes[0].set_title("Original Numer")
            axes[0].axis("off")
            axes[1].imshow(recon, cmap="gray", vmin=0, vmax=1)
            title = f"Generated (Number {label})" if label is not None else "Generation by VAE"
            axes[1].set_title(title)
            axes[1].axis("off")
        else:
            fig2, ax = plt.subplots(figsize=(3, 3))
            ax.imshow(recon, cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"latent=({x:.2f}, {y:.2f})")
            ax.axis("off")
        plt.tight_layout()
        plt.show()

def on_click_marker(trace, points, state):
    # 실제 데이터 점을 클릭한 경우 -> 원본과 재생성 이미지를 함께 표시
    if not points.point_inds:
        return
    i = points.point_inds[0]
    show_reconstruction(z[i, 0], z[i, 1], original_image=images_for_plot[i], label=labels_plot[i])

def on_click_empty(trace, points, state):
    # 빈 공간을 클릭한 경우 -> 그 좌표만으로 디코더가 이미지 생성
    if not points.point_inds:
        return
    x_click = points.xs[0]
    y_click = points.ys[0]
    show_reconstruction(x_click, y_click)

fig.data[0].on_click(on_click_empty)   # heatmap (배경)
fig.data[1].on_click(on_click_marker)  # scatter (마커)

display(VBox([fig, out]))

Epoch 1/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - kl_loss: 19.1786 - loss: 297.3126 - recon_loss: 278.1341
Epoch 2/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - kl_loss: 4.7907 - loss: 200.6060 - recon_loss: 195.8153
Epoch 3/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - kl_loss: 3.2173 - loss: 195.6719 - recon_loss: 192.4546
Epoch 4/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - kl_loss: 3.6618 - loss: 186.9657 - recon_loss: 183.3039
Epoch 5/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - kl_loss: 3.6480 - loss: 177.0424 - recon_loss: 173.3944
Epoch 6/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - kl_loss: 3.6323 - loss: 173.8959 - recon_loss: 170.2636
Epoch 7/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - kl_loss: 3.7264 - loss: 172.4357 - recon_loss: 168.7094
Epoch 8/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - kl_loss: 3.8593 - loss: 169.9608 - recon_loss: 166.1014
Epoch 9/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - kl_loss: 4.0496 - loss: 168.2160 - recon_loss: 164.1664
Epoch 10/20
45/45 

    'data': [{'colorscale': [[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
      …